# Downtown LA Weather Analysis

This notebook reads data directly from the SQLite database generated by the ingestion pipeline.

**Data source:** `../data/weather.db`, table `daily_weather` joined with `sources`.

Run `poetry run ingest-weather` before executing this notebook.

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

db_path = Path("../data/weather.db").resolve()
conn = sqlite3.connect(db_path)

query = """
SELECT
    d.observation_date,
    d.temp_max_f,
    d.temp_min_f,
    d.temp_avg_f,
    d.temp_departure,
    d.hdd,
    d.cdd,
    d.precip_inches,
    d.snow_depth,
    d.quality_flag,
    s.kind AS source_kind,
    s.filename AS source_file
FROM daily_weather d
JOIN sources s ON s.id = d.source_id
"""

df = pd.read_sql_query(query, conn, parse_dates=["observation_date"])
df["month"] = df["observation_date"].dt.to_period("M").astype(str)

print(f"Rows: {len(df)}")
df.head()

## Analysis 1: Monthly temperature trend

This chart compares monthly average maximum, minimum, and mean temperature (F).

Source: `daily_weather` from SQLite (`source_kind` includes both CSV and PDF rows).

In [ ]:
monthly_temp = (
    df.groupby("month", as_index=False)[["temp_max_f", "temp_min_f", "temp_avg_f"]]
    .mean(numeric_only=True)
)

plt.figure(figsize=(10, 5))
for col, label in [
    ("temp_max_f", "Avg Max Temp (F)"),
    ("temp_min_f", "Avg Min Temp (F)"),
    ("temp_avg_f", "Avg Mean Temp (F)"),
]:
    sns.lineplot(data=monthly_temp, x="month", y=col, marker="o", label=label)

plt.title("Downtown LA Monthly Temperature Trend")
plt.xlabel("Month")
plt.ylabel("Temperature (F)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

monthly_temp

## Analysis 2: Precipitation and wet-day distribution

This analysis shows wet days (`precip_inches > 0`) by month and the precipitation distribution on those days.

Source: `daily_weather.precip_inches` from SQLite.

In [ ]:
wet_days = df[df["precip_inches"].fillna(0) > 0].copy()
wet_counts = wet_days.groupby("month", as_index=False).size().rename(columns={"size": "wet_days"})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(data=wet_counts, x="month", y="wet_days", ax=axes[0], color="#4C78A8")
axes[0].set_title("Wet Days by Month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Days with Precipitation")
axes[0].tick_params(axis="x", rotation=30)

sns.histplot(data=wet_days, x="precip_inches", bins=12, ax=axes[1], color="#59A14F")
axes[1].set_title("Distribution of Daily Precipitation")
axes[1].set_xlabel("Precipitation (inches)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

wet_counts

## Analysis 3: Data quality and source comparison

This view quantifies rows with quality flags and compares source coverage.

Source: `daily_weather.quality_flag` and `sources.kind` from SQLite.

In [ ]:
quality_summary = (
    df.assign(has_quality_issue=df["quality_flag"].notna())
    .groupby("source_kind", as_index=False)["has_quality_issue"]
    .agg(total_issue_rows="sum")
)
source_counts = df.groupby("source_kind", as_index=False).size().rename(columns={"size": "total_rows"})
summary = source_counts.merge(quality_summary, on="source_kind", how="left")
summary["total_issue_rows"] = summary["total_issue_rows"].fillna(0).astype(int)
summary["issue_rate"] = (summary["total_issue_rows"] / summary["total_rows"]).round(3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=summary, x="source_kind", y="total_rows", ax=axes[0], color="#F28E2B")
axes[0].set_title("Rows by Source")
axes[0].set_xlabel("Source")
axes[0].set_ylabel("Row Count")

sns.barplot(data=summary, x="source_kind", y="issue_rate", ax=axes[1], color="#E15759")
axes[1].set_title("Quality-Flag Rate by Source")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Issue Rate")

plt.tight_layout()
plt.show()

summary

In [ ]:
conn.close()